In [0]:
import os
import pyodbc
import re

# Define the path to the files
path = 'learn_adb_fikrat.cc_bronze.source_queries'

# Connect to the database
conn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER=your_server;DATABASE=your_database;UID=your_username;PWD=your_password')

# Create a cursor object
cursor = conn.cursor()

# Loop through each file in the path
for filename in os.listdir(path):
    # Open the file and read its content
    with open(os.path.join(path, filename), 'r') as file:
        content = file.read()

    # Split the content into smaller sections using a regular expression
    sections = re.split(r'(GO|go)', content)

    # Loop through each section
    for i in range(0, len(sections), 2):
        # Get the original query and the section
        original_query = content
        section = sections[i] + ('GO' if i + 1 < len(sections) else '')

        # Remove leading and trailing whitespace from the section
        section = section.strip()

        # Insert the section and the original query into the database table
        cursor.execute("INSERT INTO learn_adb_fikrat.cc_bronze.source_queries (query, query_section) VALUES (?, ?)", original_query, section)

# Commit the changes and close the cursor and connection
conn.commit()
cursor.close()
conn.close()

In [0]:
sql_command="""
WITH RecentOrders AS (
    SELECT 
        o.OrderID,
        o.CustomerID,
        o.OrderDate,
        o.TotalAmount
    FROM Orders o
    WHERE o.OrderDate >= DATEADD(MONTH, -6, GETDATE())  -- last 6 months
),
TopCustomers AS (
    SELECT 
        CustomerID,
        SUM(TotalAmount) AS TotalSpent
    FROM RecentOrders
    GROUP BY CustomerID
    HAVING SUM(TotalAmount) > 5000  -- high-value customers
),
ProductSales AS (
    SELECT 
        p.ProductID,
        p.ProductName,
        SUM(od.Quantity) AS TotalQty,
        SUM(od.Quantity * od.UnitPrice) AS Revenue
    FROM OrderDetails od
    INNER JOIN Products p ON od.ProductID = p.ProductID
    GROUP BY p.ProductID, p.ProductName
),
CustomerDetails AS (
    SELECT 
        c.CustomerID,
        c.CompanyName,
        c.City,
        c.Country
    FROM Customers c
    WHERE c.Country IN ('USA', 'Canada')
)
-- Main Query: Combine high-value customers with their recent orders and top products
SELECT 
    tc.CustomerID,
    cd.CompanyName,
    cd.City,
    cd.Country,
    ro.OrderID,
    ro.OrderDate,
    ro.TotalAmount,
    ps.ProductName,
    ps.Revenue
FROM TopCustomers tc
INNER JOIN CustomerDetails cd ON tc.CustomerID = cd.CustomerID
INNER JOIN RecentOrders ro ON tc.CustomerID = ro.CustomerID
LEFT JOIN ProductSales ps ON ps.Revenue > 10000  -- only top-selling products
WHERE ro.TotalAmount > 100
UNION
-- Include customers with no recent orders but high product purchases
SELECT 
    cd.CustomerID,
    cd.CompanyName,
    cd.City,
    cd.Country,
    NULL AS OrderID,
    NULL AS OrderDate,
    NULL AS TotalAmount,
    ps.ProductName,
    ps.Revenue
FROM CustomerDetails cd
INNER JOIN ProductSales ps ON ps.Revenue > 20000
ORDER BY Revenue DESC;
SELECT * FROM CustomerDetails;
"""

In [0]:
split_statements = sqlparse.split(sql_command)
statements = {}
for split_statement in split_statements:
    tokens = sqlparse.parse(split_statement)
    print (tokens)


In [0]:
import sqlparse,os
# from core.utils.logging_config import get_logger

path = '/Volumes/learn_adb_fikrat/cc_bronze/source_queries'
for filename in os.listdir(path):

    # Open the file and read its content
    with open(os.path.join(path, filename), 'r') as file:
        content = file.read()
        split_statements = sqlparse.split(content)
        statements = {}
        for split_statement in split_statements:
            tokens = sqlparse.parse(split_statement)
            _extract_subqueries(tokens, statements)
            print('Extracted subqueries:\n %s', statements)

def _extract_subqueries(tokens: list, statements=None):
    """
    Recursively extracts subqueries from SQL tokens.
    
    This function traverses the SQL token tree and identifies SELECT statements
    inside parentheses, which are typically subqueries.
    
    Args:
        tokens (list): SQL tokens parsed by sqlparse
        statements (dict, optional): Dictionary to store extracted subqueries. Defaults to None.
        
    Returns:
        tuple: (token_string, subqueries_list) where:
            - token_string is the string representation of the token
            - subqueries_list is a list of extracted subqueries
    """
    if statements is None:
        statements = {}
    if type(tokens) != sqlparse.sql.Token:
        parenthesis = isinstance(tokens, sqlparse.sql.Parenthesis)
        v = [_extract_subqueries(i, statements) for i in (tokens if not parenthesis else tokens)]
        tokens_subset, qrs = ''.join(str(i[0]) for i in v), [x for _, y in v for x in y]
        if [*tokens][parenthesis].value.upper() == 'SELECT':
            sub_count = len(statements)
            sub_count += 1
            statements[f'subquery_{sub_count}'] = [tokens_subset]+qrs
            return f'subquery_{sub_count}', [tokens_subset]+qrs
        return tokens_subset, qrs
    return tokens, []

def _replace_subqueries_with_placeholders(source_code:str , expanded_subqueries:dict):
    """
    Replaces subqueries in the source code with placeholder identifiers.
    
    This allows the main SQL to be chunked without splitting subqueries.
    
    Args:
        source_code (str): Original SQL source code
        expanded_subqueries (dict): Dictionary of subqueries with their keys
        
    Returns:
        str: SQL source code with subqueries replaced by placeholders
        
    Raises:
        AssertionError: If a subquery is still present after replacement
    """
    print('Resolving placeholders in source_code')

    for key, _ in sorted(list(expanded_subqueries.items()), key=lambda x:x[0].lower(), reverse=True):
        flattned_subquery = clean_whitespace(expanded_subqueries[key][0])
        source_code = source_code.replace(flattned_subquery, key)
        assert(flattned_subquery not in source_code)
    return source_code     

In [0]:
multi_sql = "SELECT 1; INSERT INTO other_table VALUES (2);"
statements = sqlparse.split(multi_sql)
for stmt in statements:
        print(stmt)